# MatRisk AI — Notebook 00: Dataset Inventory and Quality Audit

This notebook profiles the supplied six CSV files and workbook. It does not train models. Findings are diagnostic flags, not automatic proof that every flagged row is wrong. Keep the original files unchanged and record any remediation separately.

In [8]:

from pathlib import Path
import os
import pandas as pd
import numpy as np
import json

# Search locations relative to the Jupyter working directory
DATA_DIR_CANDIDATES = [
    # Optional explicit override: set MATRISK_DATA_ROOT to your Data folder.
    Path(os.environ["MATRISK_DATA_ROOT"]) if os.environ.get("MATRISK_DATA_ROOT") else Path("__MATRISK_NO_OVERRIDE__"),
    Path("Data"),
    Path("Data/raw"),
    Path("../Data"),
    Path("../Data/raw"),
    Path("../../Data"),
    Path("../../Data/raw"),
    Path("data/raw"),
    Path("data/external"),
    Path(".."),
    # Hosted notebook fallback; harmless on a local Windows setup.
    Path("/mnt/data"),
]

FILES = {
    "DS1": "DS1_material_properties_5500.csv",
    "DS2": "DS2_commodity_prices_10yr.csv",
    "DS3": "DS3_infrastructure_bridges_5000.csv",
    "DS4": "DS4_crossdomain_features_daily.csv",
    "DS5": "DS5_element_prices_monthly.csv",
    "DS6": "DS6_historical_failures_2000.csv",
}

def find_file(filename):
    for directory in DATA_DIR_CANDIDATES:
        candidate = directory / filename
        if candidate.is_file():
            print(f"Found {filename}: {candidate.resolve()}")
            return candidate

    searched = "\n".join(
        f" - {p.resolve()}" for p in DATA_DIR_CANDIDATES
    )
    raise FileNotFoundError(
        f"Missing {filename}. Searched:\n{searched}"
    )

loaded = {
    key: pd.read_csv(find_file(filename))
    for key, filename in FILES.items()
}

for key, frame in loaded.items():
    print(f"{key}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")


Found DS1_material_properties_5500.csv: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\Data\raw\DS1_material_properties_5500.csv
Found DS2_commodity_prices_10yr.csv: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\Data\raw\DS2_commodity_prices_10yr.csv
Found DS3_infrastructure_bridges_5000.csv: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\Data\raw\DS3_infrastructure_bridges_5000.csv
Found DS4_crossdomain_features_daily.csv: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\Data\raw\DS4_crossdomain_features_daily.csv
Found DS5_element_prices_monthly.csv: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\Data\raw\DS5_element_prices_monthly.csv
Found DS6_historical_failures_2000.csv: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\Data\raw\DS6_historical_failures_2000.csv
DS1: 5,500 rows × 18 columns
DS2: 22,952 rows × 24 columns
DS3: 5,000 rows × 24 columns
DS4: 22,952 rows × 12 columns
DS5: 5,280 rows × 5 columns
DS6: 2,000 rows × 18 columns


In [9]:
inventory_rows = []
for key, frame in loaded.items():
    inventory_rows.append({
        "dataset": key, "rows": len(frame), "columns": frame.shape[1],
        "missing_cells": int(frame.isna().sum().sum()),
        "duplicate_full_rows": int(frame.duplicated().sum()),
        "memory_mb": round(frame.memory_usage(deep=True).sum()/1024**2, 2),
    })
inventory = pd.DataFrame(inventory_rows)
display(inventory)


,dataset,rows,columns,missing_cells,duplicate_full_rows,memory_mb
0,DS1,5500,18,0,0,1.78
1,DS2,22952,24,2760,0,6.42
2,DS3,5000,24,0,0,2.40
3,DS4,22952,12,712,0,4.32
4,DS5,5280,5,0,0,0.67
5,DS6,2000,18,0,0,1.06


In [10]:
# Dataset-specific integrity checks
d1, d2, d3, d4, d5, d6 = (loaded[k] for k in ["DS1","DS2","DS3","DS4","DS5","DS6"])
findings = []
def add(dataset, check, value, interpretation):
    findings.append({"dataset": dataset, "check": check, "value": value, "interpretation": interpretation})

add("DS1", "duplicate material_id rows", int(d1.material_id.duplicated().sum()), "Investigate IDs; repeated IDs can invalidate row-level assumptions.")
nu = d1.poisson_ratio
add("DS1", "Poisson ratio outside (-1, 0.5)", int(((nu <= -1) | (nu >= 0.5)).sum()), "Physics-bound check.")
add("DS1", "nonpositive K or G", int(((d1.bulk_modulus_GPa <= 0) | (d1.shear_modulus_GPa <= 0)).sum()), "Physics-bound check.")
eg = 2*d1.shear_modulus_GPa*(1+nu)
ek = 3*d1.bulk_modulus_GPa*(1-2*nu)
rel = np.abs(eg-ek)/np.maximum(np.maximum(np.abs(eg),np.abs(ek)),1e-12)
add("DS1", "elastic relation discrepancy >5%", int((rel > .05).sum()), "E inferred from K and G/nu is inconsistent beyond 5%; review tolerance and source quality.")
add("DS1", "stable label with formation energy >= 0", int(((d1.is_stable==1)&(d1.formation_energy_per_atom_eV>=0)).sum()), "Check label definition and energy reference before using as a rule.")
add("DS2", "duplicate (date, commodity)", int(d2.duplicated(["date","commodity"]).sum()), "Must be zero before time-series merge.")
add("DS2", "date range", f"{d2.date.min()} to {d2.date.max()}", "Verify chronological ordering and source provenance.")
add("DS3", "condition_rating == 0", int((d3.condition_rating==0).sum()), "README states rating range 1–9; zero requires investigation.")
add("DS3", "bridge IDs duplicated", int(d3.bridge_id.duplicated().sum()), "Must be zero for unique bridge rows.")
add("DS3", "event time/censoring columns available", False, "A single snapshot without observed event time and censoring cannot directly validate survival analysis.")
add("DS4", "duplicate (date, commodity)", int(d4.duplicated(["date","commodity"]).sum()), "Must be zero before merge.")
add("DS4", "keys not present in DS2", int((~pd.MultiIndex.from_frame(d4[["date","commodity"]]).isin(pd.MultiIndex.from_frame(d2[["date","commodity"]]))).sum()), "Cross-domain merge keys should align.")
add("DS5", "duplicate (date, element)", int(d5.duplicated(["date","element"]).sum()), "Must be zero for monthly element-price panel.")
add("DS6", "unique event IDs", int(d6.event_id.nunique()), "Event-only records are not a full exposed population.")
add("DS6", "rows where repair cost exceeds replacement value", int((d6.repair_cost_USD>d6.replacement_value_USD).sum()), "Review outliers and definition; do not silently cap.")
findings_df = pd.DataFrame(findings)
display(findings_df)


,dataset,check,value,interpretation
0,DS1,duplicate material_id rows,21,Investigate IDs; repeated IDs can invalidate r...
1,DS1,"Poisson ratio outside (-1, 0.5)",0,Physics-bound check.
2,DS1,nonpositive K or G,0,Physics-bound check.
3,DS1,elastic relation discrepancy >5%,1765,E inferred from K and G/nu is inconsistent bey...
4,DS1,stable label with formation energy >= 0,369,Check label definition and energy reference be...
5,DS2,"duplicate (date, commodity)",0,Must be zero before time-series merge.
6,DS2,date range,2014-01-02 to 2024-12-31,Verify chronological ordering and source prove...
7,DS3,condition_rating == 0,5,README states rating range 1–9; zero requires ...
8,DS3,bridge IDs duplicated,0,Must be zero for unique bridge rows.
9,DS3,event time/censoring columns available,False,A single snapshot without observed event time ...


In [11]:
# Cross-domain key integrity and workbook sheet inventory
keys2 = pd.MultiIndex.from_frame(d2[["date","commodity"]])
keys4 = pd.MultiIndex.from_frame(d4[["date","commodity"]])
print("DS2 keys:", len(keys2), "DS4 keys:", len(keys4), "exact key-set match:", set(keys2)==set(keys4))
xlsx_candidates = [p/"MatRisk_AI_Sample_Datasets.xlsx" for p in DATA_DIR_CANDIDATES]
xlsx_path = next((p for p in xlsx_candidates if p.exists()), None)
if xlsx_path:
    xl = pd.ExcelFile(xlsx_path)
    print("Workbook:", xlsx_path)
    print("Sheets:", xl.sheet_names)
else:
    print("Workbook not found; CSV audit completed.")


DS2 keys: 22952 DS4 keys: 22952 exact key-set match: True
Workbook: ..\..\Data\raw\MatRisk_AI_Sample_Datasets.xlsx
Sheets: ['DS1_Material_Properties', 'DS2_Commodity_Prices', 'DS3_Infrastructure_Portfolio', 'DS4_CrossDomain_Features', 'DS5_Stress_Scenarios', 'Data_Dictionary']


In [12]:
OUTPUT_DIR = Path("outputs/data_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
inventory.to_csv(OUTPUT_DIR/"dataset_inventory.csv", index=False)
findings_df.to_csv(OUTPUT_DIR/"quality_findings.csv", index=False)
print("Saved audit outputs to", OUTPUT_DIR.resolve())


Saved audit outputs to C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\ML\notebooks\outputs\data_audit


Modelling implications
- DS-1: begin with composition-based regression baselines and grouped evaluation. The CSV does not contain atomic coordinates/crystal structures, so a crystal GNN is not currently supported.
- DS-2 + DS-4: use chronological walk-forward evaluation. Construct targets by shifting future returns; do not include same-day close or future-derived features in predictors.
- DS-3: a bridge snapshot may support exploratory condition/risk proxies, but it does not by itself provide a valid time-to-event cohort.
- DS-5: verify element coverage and monthly price semantics before using as a cost constraint.
- DS-6: event-only rows cannot estimate population failure probability without exposure/non-failure observations.
- Preserve all original values. Keep cleaning decisions in separate derived tables and document every transformation